# Pokemon Complete Stats (Gen 1 to Gen 9) - EDA

Exploratory Data Analysis of 1,025 Pokemon across all 9 generations. The dataset includes base stats (HP, Attack, Defense, Sp. Attack, Sp. Defense, Speed), types, physical characteristics, and abilities.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

# Type color palette
TYPE_COLORS = {
    'normal': '#A8A878', 'fire': '#F08030', 'water': '#6890F0', 'electric': '#F8D030',
    'grass': '#78C850', 'ice': '#98D8D8', 'fighting': '#C03028', 'poison': '#A040A0',
    'ground': '#E0C068', 'flying': '#A890F0', 'psychic': '#F85888', 'bug': '#A8B820',
    'rock': '#B8A038', 'ghost': '#705898', 'dragon': '#7038F8', 'dark': '#705848',
    'steel': '#B8B8D0', 'fairy': '#EE99AC'
}

## 1. Data Loading & Overview

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/ibrahimqasimi/pokemon-complete-stats-gen-1-to-gen-9/Pokemon_Complete_Gen1_to_Gen9.csv')

# Assign generation based on Pokedex number
gen_bounds = [(1, 151), (152, 251), (252, 386), (387, 493), (494, 649), (650, 721), (722, 809), (810, 905), (906, 1025)]
for i, (lo, hi) in enumerate(gen_bounds, 1):
    df.loc[(df['id'] >= lo) & (df['id'] <= hi), 'generation'] = i
df['generation'] = df['generation'].astype(int)

# Total base stats
df['total_stats'] = df[['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed']].sum(axis=1)

# Single vs dual type
df['is_dual_type'] = df['type2'].notna()

print(f'Shape: {df.shape}')
print(f'Generations: {df["generation"].nunique()} (Gen {df["generation"].min()} to Gen {df["generation"].max()})')
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
df.head()

In [ ]:
df.describe().round(2)

## 2. Type Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Primary type distribution
type1_counts = df['type1'].value_counts()
colors1 = [TYPE_COLORS.get(t, '#888888') for t in type1_counts.index]
axes[0].barh(type1_counts.index, type1_counts.values, color=colors1, edgecolor='white')
axes[0].set_title('Primary Type Distribution')
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()
for i, v in enumerate(type1_counts.values):
    axes[0].text(v + 1, i, str(v), va='center', fontsize=9)

# Secondary type distribution
type2_counts = df['type2'].value_counts()
colors2 = [TYPE_COLORS.get(t, '#888888') for t in type2_counts.index]
axes[1].barh(type2_counts.index, type2_counts.values, color=colors2, edgecolor='white')
axes[1].set_title('Secondary Type Distribution')
axes[1].set_xlabel('Count')
axes[1].invert_yaxis()
for i, v in enumerate(type2_counts.values):
    axes[1].text(v + 1, i, str(v), va='center', fontsize=9)

plt.tight_layout()
plt.show()

dual_pct = df['is_dual_type'].mean() * 100
print(f'Dual-type Pokemon: {df["is_dual_type"].sum()} ({dual_pct:.1f}%)')
print(f'Single-type Pokemon: {(~df["is_dual_type"]).sum()} ({100 - dual_pct:.1f}%)')

In [ ]:
# Type combination heatmap
type_combos = df.groupby(['type1', 'type2']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(type_combos, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Type Combination Heatmap (Primary x Secondary)')
ax.set_xlabel('Secondary Type')
ax.set_ylabel('Primary Type')
plt.tight_layout()
plt.show()

## 3. Base Stats Distribution

In [ ]:
stat_cols = ['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(stat_cols):
    ax = axes[i // 3, i % 3]
    ax.hist(df[col], bins=30, alpha=0.7, color='steelblue', edgecolor='white')
    ax.axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.0f}')
    ax.axvline(df[col].median(), color='orange', linestyle='--', label=f'Median: {df[col].median():.0f}')
    ax.set_title(col.replace('_', ' ').title())
    ax.legend(fontsize=9)

plt.suptitle('Base Stats Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Total base stats distribution
fig, ax = plt.subplots(figsize=(14, 5))
ax.hist(df['total_stats'], bins=40, alpha=0.7, color='steelblue', edgecolor='white')
ax.axvline(df['total_stats'].mean(), color='red', linestyle='--', label=f'Mean: {df["total_stats"].mean():.0f}')
ax.axvline(600, color='purple', linestyle='--', alpha=0.7, label='Pseudo-legendary threshold (600)')
ax.set_title('Total Base Stats Distribution')
ax.set_xlabel('Total Base Stats')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

pseudo_legendary = df[df['total_stats'] >= 600].sort_values('total_stats', ascending=False)
print(f'Pokemon with 600+ total stats: {len(pseudo_legendary)}')
print(pseudo_legendary[['name', 'type1', 'type2', 'total_stats']].head(10).to_string(index=False))

## 4. Stats by Type

In [ ]:
type_stats = df.groupby('type1')[stat_cols + ['total_stats']].mean().sort_values('total_stats', ascending=False)

fig, ax = plt.subplots(figsize=(14, 7))
type_order = type_stats.index
colors = [TYPE_COLORS.get(t, '#888888') for t in type_order]
type_stats[stat_cols].plot(kind='bar', stacked=True, ax=ax, width=0.8,
                           color=['#FF6B6B', '#FF8E53', '#FFC93C', '#6BCB77', '#4D96FF', '#9B59B6'])
ax.set_title('Average Base Stats by Primary Type (sorted by total)')
ax.set_ylabel('Stat Value')
ax.set_xlabel('Primary Type')
ax.legend(title='Stat', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Box plots of total stats by type
fig, ax = plt.subplots(figsize=(16, 6))
type_order = df.groupby('type1')['total_stats'].median().sort_values(ascending=False).index
palette = {t: TYPE_COLORS.get(t, '#888888') for t in type_order}
sns.boxplot(data=df, x='type1', y='total_stats', order=type_order, palette=palette, ax=ax)
ax.set_title('Total Base Stats Distribution by Primary Type')
ax.set_xlabel('Primary Type')
ax.set_ylabel('Total Base Stats')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Generation Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Pokemon count per generation
gen_counts = df['generation'].value_counts().sort_index()
axes[0].bar(gen_counts.index, gen_counts.values, color='steelblue', alpha=0.7)
axes[0].set_title('Pokemon Count by Generation')
axes[0].set_xlabel('Generation')
axes[0].set_ylabel('Count')
axes[0].set_xticks(range(1, 10))
for i, v in enumerate(gen_counts.values):
    axes[0].text(gen_counts.index[i], v + 1, str(v), ha='center', fontsize=9)

# Average total stats by generation
gen_stats = df.groupby('generation')['total_stats'].mean()
axes[1].bar(gen_stats.index, gen_stats.values, color='coral', alpha=0.7)
axes[1].set_title('Average Total Base Stats by Generation')
axes[1].set_xlabel('Generation')
axes[1].set_ylabel('Avg Total Stats')
axes[1].set_xticks(range(1, 10))
axes[1].axhline(df['total_stats'].mean(), color='red', linestyle='--', alpha=0.5, label='Overall mean')
axes[1].legend()

# Dual type percentage by generation
gen_dual = df.groupby('generation')['is_dual_type'].mean() * 100
axes[2].bar(gen_dual.index, gen_dual.values, color='mediumpurple', alpha=0.7)
axes[2].set_title('Dual-Type Percentage by Generation')
axes[2].set_xlabel('Generation')
axes[2].set_ylabel('Dual-Type (%)')
axes[2].set_xticks(range(1, 10))

plt.tight_layout()
plt.show()

In [ ]:
# Type distribution evolution across generations
gen_type = df.groupby(['generation', 'type1']).size().unstack(fill_value=0)
gen_type_pct = gen_type.div(gen_type.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 7))
gen_type_pct.plot(kind='bar', stacked=True, ax=ax, width=0.8,
                  color=[TYPE_COLORS.get(t, '#888888') for t in gen_type_pct.columns])
ax.set_title('Primary Type Distribution by Generation (%)')
ax.set_xlabel('Generation')
ax.set_ylabel('Percentage')
ax.legend(title='Type', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## 6. Physical Characteristics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Height distribution
axes[0].hist(df['height'] / 10, bins=40, alpha=0.7, color='steelblue', edgecolor='white')
axes[0].set_title('Height Distribution')
axes[0].set_xlabel('Height (m)')
axes[0].set_ylabel('Count')

# Weight distribution
axes[1].hist(df['weight'] / 10, bins=40, alpha=0.7, color='coral', edgecolor='white')
axes[1].set_title('Weight Distribution')
axes[1].set_xlabel('Weight (kg)')
axes[1].set_ylabel('Count')

# Height vs Weight scatter colored by type
for t in df['type1'].unique():
    subset = df[df['type1'] == t]
    axes[2].scatter(subset['height'] / 10, subset['weight'] / 10, s=15, alpha=0.5,
                    color=TYPE_COLORS.get(t, '#888888'), label=t)
axes[2].set_title('Height vs Weight by Type')
axes[2].set_xlabel('Height (m)')
axes[2].set_ylabel('Weight (kg)')
axes[2].set_xscale('log')
axes[2].set_yscale('log')

plt.tight_layout()
plt.show()

# Extremes
for col, label, scale in [('height', 'Height', 10), ('weight', 'Weight', 10)]:
    top = df.nlargest(3, col)[['name', col]].copy()
    top[col] = top[col] / scale
    unit = 'm' if col == 'height' else 'kg'
    print(f'Tallest/Heaviest ({label}): {", ".join(f"{r["name"]} ({r[col]}{unit})" for _, r in top.iterrows())}')

## 7. Correlation Analysis

In [ ]:
corr_cols = ['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed',
             'height', 'weight', 'base_experience', 'total_stats']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 8. Attack vs Defense Profile

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Physical: Attack vs Defense
for t in df['type1'].unique():
    subset = df[df['type1'] == t]
    axes[0].scatter(subset['attack'], subset['defense'], s=15, alpha=0.4,
                    color=TYPE_COLORS.get(t, '#888888'))
axes[0].plot([0, 250], [0, 250], 'k--', alpha=0.3, label='Equal line')
axes[0].set_title('Attack vs Defense')
axes[0].set_xlabel('Attack')
axes[0].set_ylabel('Defense')
axes[0].legend()

# Special: Sp.Atk vs Sp.Def
for t in df['type1'].unique():
    subset = df[df['type1'] == t]
    axes[1].scatter(subset['sp_attack'], subset['sp_defense'], s=15, alpha=0.4,
                    color=TYPE_COLORS.get(t, '#888888'))
axes[1].plot([0, 250], [0, 250], 'k--', alpha=0.3, label='Equal line')
axes[1].set_title('Sp. Attack vs Sp. Defense')
axes[1].set_xlabel('Sp. Attack')
axes[1].set_ylabel('Sp. Defense')
axes[1].legend()

plt.tight_layout()
plt.show()

# Offensive vs Defensive Pokemon
df['offense'] = df['attack'] + df['sp_attack']
df['defense_total'] = df['defense'] + df['sp_defense']
df['off_def_ratio'] = df['offense'] / df['defense_total']

print('Most offensive Pokemon (highest atk+sp_atk / def+sp_def):')
print(df.nlargest(5, 'off_def_ratio')[['name', 'type1', 'attack', 'sp_attack', 'defense', 'sp_defense', 'off_def_ratio']].to_string(index=False))
print('\nMost defensive Pokemon:')
print(df.nsmallest(5, 'off_def_ratio')[['name', 'type1', 'attack', 'sp_attack', 'defense', 'sp_defense', 'off_def_ratio']].to_string(index=False))

## 9. Radar Charts - Iconic Pokemon Comparison

In [ ]:
def radar_chart(ax, pokemon_data, name, color):
    categories = ['HP', 'Attack', 'Defense', 'Sp.Atk', 'Sp.Def', 'Speed']
    N = len(categories)
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]
    values = pokemon_data.tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, color=color, label=name)
    ax.fill(angles, values, alpha=0.15, color=color)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=9)
    ax.set_ylim(0, 260)

# Compare starters final evolutions + Pikachu
pokemon_names = ['charizard', 'blastoise', 'venusaur', 'pikachu', 'mewtwo', 'garchomp']
colors_radar = ['#F08030', '#6890F0', '#78C850', '#F8D030', '#F85888', '#7038F8']

fig, axes = plt.subplots(2, 3, figsize=(16, 12), subplot_kw=dict(projection='polar'))
for i, (name, color) in enumerate(zip(pokemon_names, colors_radar)):
    ax = axes[i // 3, i % 3]
    poke = df[df['name'] == name]
    if len(poke) > 0:
        vals = poke[stat_cols].values[0]
        radar_chart(ax, vals, name.title(), color)
        ax.set_title(f'{name.title()}\n(Total: {poke["total_stats"].values[0]})', fontsize=11, pad=20)

plt.suptitle('Stat Radar Charts - Iconic Pokemon', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 10. Abilities Analysis

In [ ]:
# Split abilities and count
all_abilities = df['abilities'].str.split(', ').explode()
ability_counts = all_abilities.value_counts()

fig, ax = plt.subplots(figsize=(14, 6))
top_abilities = ability_counts.head(20)
ax.barh(top_abilities.index[::-1], top_abilities.values[::-1], color='steelblue', alpha=0.7)
ax.set_title('Top 20 Most Common Abilities')
ax.set_xlabel('Number of Pokemon')
for i, v in enumerate(top_abilities.values[::-1]):
    ax.text(v + 0.3, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.show()

# Ability count per Pokemon
df['ability_count'] = df['abilities'].str.split(', ').apply(len)
print(f'Ability count distribution:')
print(df['ability_count'].value_counts().sort_index().to_string())
print(f'\nTotal unique abilities: {ability_counts.nunique()}')

## 11. Stat Leaders

In [ ]:
print('=== TOP 3 IN EACH STAT ===\n')
for stat in stat_cols:
    top3 = df.nlargest(3, stat)[['name', 'type1', stat]]
    label = stat.replace('_', ' ').title()
    print(f'{label}:')
    for _, row in top3.iterrows():
        print(f'  {row["name"]:20s} ({row["type1"]:8s}) - {row[stat]}')
    print()

print('=== TOP 10 OVERALL (Total Base Stats) ===\n')
print(df.nlargest(10, 'total_stats')[['name', 'type1', 'type2'] + stat_cols + ['total_stats']].to_string(index=False))

## 12. Key Findings

Summary of the exploratory analysis of the Pokemon dataset (Gen 1 to Gen 9):

- **Type balance**: Water is the most common primary type (134), while Flying dominates as a secondary type (100). About 51% of Pokemon are dual-type
- **Stat distributions**: All six base stats follow roughly normal distributions with right skew from legendary/pseudo-legendary outliers. The average total base stat is ~428
- **Type strengths**: Dragon-type Pokemon have the highest average total stats, while Bug-type tend to be the weakest statistically
- **Generation trends**: Gen 5 introduced the most new Pokemon (156), while later generations have been more conservative. Average stats remain fairly consistent across generations
- **Physical characteristics**: Height and weight distributions are heavily right-skewed, with a few extreme outliers (Wailord, Celesteela, etc.)
- **Correlations**: Base experience strongly correlates with total stats. Attack and Sp. Attack are weakly correlated, suggesting most Pokemon specialize in one offensive stat
- **Offensive vs Defensive**: Clear specialization exists — some Pokemon are glass cannons (high offense, low defense) while others are pure walls
- **Abilities**: Levitate is the most common ability. Most Pokemon have 2 abilities, with some having up to 3 (including hidden abilities)